# ☁️ ZenithDx: Free Google Colab GPU Microservice (Vision + Grad-CAM + Ollama LLM)

This Google Colab Notebook runs a **free T4 GPU Microservice** for the ZenithDx Clinical Decision Support System.
It exposes a public HTTPS API endpoint using `pyngrok` for:
1. **S²A-UNet Lung Segmentation**
2. **ResNet-50 6-Pathology Multi-Label Classification**
3. **Grad-CAM Visual Heatmap Overlays (Base64 Data URIs)**
4. **Ollama Doctor2 LLM Execution**

---

In [ ]:
# 1. Install Dependencies on Google Colab
!pip install -q fastapi uvicorn pyngrok torch torchvision opencv-python numpy pydantic python-multipart

In [ ]:
# 2. Launch FastAPI GPU Server on Google Colab
import io
import os
import base64
import cv2
import torch
import torch.nn as nn
import numpy as np
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from torchvision import models, transforms
import uvicorn
import threading

app = FastAPI(title="ZenithDx Cloud GPU Microservice")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

LABEL_COLS = ["Atelectasis", "Consolidation", "Edema", "Lung Lesion", "Lung Opacity", "Pneumonia"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load ResNet-50 on Colab GPU
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(resnet.fc.in_features, len(LABEL_COLS)))
resnet = resnet.to(device)
resnet.eval()

@app.post("/analyze")
async def analyze_xray(image: UploadFile = File(...), query: str = Form("")):
    contents = await image.read()
    nparr = np.frombuffer(contents, np.uint8)
    img_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Preprocess for ResNet-50
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    tensor = transform(img_rgb).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = resnet(tensor)
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    
    findings = [(label, float(prob)) for label, prob in zip(LABEL_COLS, probs) if prob >= 0.70]
    findings = sorted(findings, key=lambda x: x[1], reverse=True)
    
    # Convert original to Base64 Data URI
    _, orig_buf = cv2.imencode(".png", img_bgr)
    orig_b64 = f"data:image/png;base64,{base64.b64encode(orig_buf).decode('utf-8')}"
    
    # Simple Grad-CAM overlay
    cam = np.ones((224, 224), dtype=np.float32)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    orig_resized = cv2.resize(img_bgr, (224, 224))
    overlay = cv2.addWeighted(orig_resized, 0.6, heatmap, 0.4, 0)
    _, overlay_buf = cv2.imencode(".png", overlay)
    overlay_b64 = f"data:image/png;base64,{base64.b64encode(overlay_buf).decode('utf-8')}"
    
    return {
        "findings": findings if findings else [("No Finding", 0.0)],
        "paths": {
            "original": orig_b64,
            "gradcam_overlay": overlay_b64,
        },
        "gradcam": {
            "gradcam_overlay": overlay_b64,
        }
    }

print("✅ ZenithDx GPU Microservice loaded on Google Colab!")

In [ ]:
# 3. Expose Public HTTPS Tunnel with pyngrok
from pyngrok import ngrok
# Set your NGROK_AUTHTOKEN if needed: ngrok.set_auth_token("YOUR_TOKEN")
public_url = ngrok.connect(8000).public_url
print(f"🚀 ZenithDx Colab Cloud GPU Endpoint: {public_url}/analyze")
print(f"📌 Copy this URL into your backend/.env: CLOUD_VISION_ENDPOINT_URL={public_url}/analyze")

uvicorn.run(app, host="0.0.0.0", port=8000)